# Learning Edge Detection with a Minimal CNN

This notebook demonstrates how a convolutional neural network can **learn** to detect edges
in images, discovering filters similar to classical hand-designed operators like the Sobel filter.

## What You'll Learn

1. **Convolution as pattern matching** — How sliding a small kernel over an image detects local patterns
2. **The Sobel operator** — A classical hand-designed edge detector based on image gradients
3. **The Beucher gradient** — A morphological alternative that's robust to noise
4. **Learning filters from data** — How gradient descent discovers edge-detecting kernels automatically
5. **Frequency response analysis** — Understanding what filters do in the frequency domain

## The Key Insight

Instead of hand-designing filters (like Sobel), we let the network **learn** what patterns
are useful. With just 18 parameters, our minimal CNN discovers gradient operators that
work as well as carefully engineered classical methods.

## Network Architecture

```
Input Image (1×H×W)
       ↓
   Conv2d(1→2, 3×3)     ← Two learnable 3×3 filters (no bias)
       ↓
   gx, gy (2×H×W)       ← Signed gradient estimates (like Sobel-X, Sobel-Y)
       ↓
   |gx| + |gy|          ← Gradient magnitude (L1 norm)
       ↓
   scale & clamp        ← Output in [0, 1]
       ↓
   Edge Map (1×H×W)
```

Only **18 parameters**: 2 filters × 9 weights, no bias.

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage.transform import resize
from skimage.morphology import dilation, erosion, square
from scipy.ndimage import correlate as ndcorrelate
from tqdm.auto import tqdm
import warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")

## 7. The Model: A Minimal Edge-Detecting CNN

Our network is intentionally minimal to make the learned filters directly interpretable.

### Architecture

```
Input (1×H×W)
     ↓
Conv2d(1→2, 3×3, padding=1)    ← Two learnable 3×3 filters (no bias)
     ↓
gx, gy (2×H×W)                 ← Signed filter responses
     ↓
|gx| + |gy|                    ← Gradient magnitude (L1 norm)
     ↓
clamp(mag / MAG_SCALE, 0, 1)   ← Scale to [0, 1]
     ↓
Output (1×H×W)
```

### Why This Architecture?

1. **Two filters** — We expect to learn two gradient directions (like Sobel-X and Sobel-Y)
2. **3×3 kernels** — Same size as Sobel, captures local gradients
3. **No activation function** — Raw filter outputs can be positive or negative (signed gradients)
4. **No bias** — Flat (constant) regions produce exactly zero response
5. **L1 magnitude** — Combines both directions: $|g_x| + |g_y|$
6. **Only 18 parameters** — 2 filters × 9 weights, easy to visualize and interpret

### What We Expect to Learn

If our network learns correctly, the two 3×3 filters should resemble gradient operators
that respond to edges in two different (ideally orthogonal) directions. They might look
like Sobel-X and Sobel-Y, or rotated versions (like 45° diagonals).

In [ ]:
# =============================================================================
# DATA CONFIGURATION
# =============================================================================
DATA_DIR = Path("./../data")  # Folder containing training images (grayscale JPEGs)
PATCH_SIZE = 512                  # Size of random patches extracted from images

# =============================================================================
# TRAINING HYPERPARAMETERS
# =============================================================================
BATCH_SIZE = 64                   # Number of patches per gradient update
LEARNING_RATE = 1e-3              # Step size for optimizer (Adam)
NUM_EPOCHS = 150                  # Number of passes through the training data
PATCHES_PER_EPOCH = 1024          # How many random patches to sample per epoch

# =============================================================================
# BEUCHER GRADIENT PARAMETERS
# =============================================================================
# The Beucher gradient is a morphological edge detector: dilation - erosion
# It's more robust to noise than simple derivatives.
SE_SIZE = 3                       # Structuring element size (3×3 square)
BEUCHER_CLIP = 1.0                # Clip gradient values to [0, BEUCHER_CLIP], then normalize

# =============================================================================
# DATA AUGMENTATION
# =============================================================================
ADD_INPUT_NOISE = True            # Add Gaussian noise to inputs (helps generalization)
NOISE_STD = 0.02                  # Standard deviation of input noise

# =============================================================================
# MODEL PARAMETERS
# =============================================================================
MAG_MODE = "l1"                   # "l1" = |gx| + |gy|, "l2" = sqrt(gx² + gy²)
MAG_SCALE = 1.0                   # Divide magnitude by this before clamping to [0,1]
CONV_BIAS = False                 # Disable bias so flat regions → zero response

# =============================================================================
# LOSS FUNCTION
# =============================================================================
LOSS_MODE = "dice"                # "l1" = Mean Absolute Error (sharper edges)
                                  # "mse" = Mean Squared Error (smoother, penalizes large errors more)
                                  # "dice" = Soft Dice (overlap-based, handles edge/background imbalance)

# =============================================================================
# TARGET MODE
# =============================================================================
TARGET_MODE = "beucher"           # "beucher" = morphological gradient (dilation − erosion)
                                  # "sobel"   = Sobel magnitude |Gx| + |Gy|

# =============================================================================
# DISPLAY
# =============================================================================
BG = '#0E2E32'                       # notebook background colour
DARK_LAYOUT = dict(paper_bgcolor=BG, plot_bgcolor=BG, font_color='white')

CMAP_EDGES_DIV = [                   # diverging: signed gradients
    [0.0, '#d4702a'],               #   negative → warm orange
    [0.5, '#111111'],               #   zero     → near-black
    [1.0, '#2ec4b6'],               #   positive → teal
]
CMAP_EDGES_SEQ = [                   # sequential: edge magnitude
    [0.0, '#111111'],               #   zero     → near-black
    [0.4, '#1a7a7a'],               #   weak     → dark teal
    [1.0, '#4de8d4'],               #   strong   → bright cyan
]

from matplotlib.colors import LinearSegmentedColormap
MPL_EDGES_DIV = LinearSegmentedColormap.from_list(
    'edges_div', ['#d4702a', '#111111', '#2ec4b6'])

def _square_sync(fig, n_cols):
    """Square pixels on first subplot, synced zoom on the rest."""
    axes = dict(
        yaxis=dict(scaleanchor='x', scaleratio=1, autorange='reversed',
                   constrain='domain'),
        xaxis=dict(constrain='domain'),
    )
    for c in range(2, n_cols + 1):
        axes[f'xaxis{c}'] = dict(constrain='domain', matches='x')
        axes[f'yaxis{c}'] = dict(autorange='reversed', constrain='domain',
                                  matches='y')
    fig.update_layout(**axes)

In [ ]:
# Select the best available compute device
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Using MPS (Apple Silicon GPU)")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("Using CUDA GPU")
else:
    DEVICE = torch.device("cpu")
    print("Using CPU")

print(f"PyTorch version: {torch.__version__}")

## 3. Understanding the Target: The Beucher Gradient

Before training a network, we need ground truth edge maps. We use the **Beucher gradient**
(also called the morphological gradient), defined as:

$\displaystyle M(x, y) = \text{dilation}(I, S) - \text{erosion}(I, S)$

Where $S$ is a structuring element (we use a 3×3 square, matching the network's 3×3 receptive field).

### Why Beucher instead of Sobel?

| Property | Sobel | Beucher |
|----------|-------|---------|
| Type | Derivative-based | Morphological |
| Noise sensitivity | High | Lower |
| Edge thickness | Thin (1-2 px) | Thicker (depends on SE size) |
| Computation | Linear filtering | Min/max operations |

The Beucher gradient highlights regions where pixel values change rapidly, similar to
classical edge detectors but with better noise robustness.

In [ ]:
def beucher_gradient(img: np.ndarray, se_size: int = 5, clip_val: float = 1.0) -> np.ndarray:
    """
    Compute the Beucher (morphological) gradient of an image.

    The Beucher gradient is defined as: dilation(img) - erosion(img)
    This highlights regions where pixel values change rapidly (edges).

    Parameters
    ----------
    img : np.ndarray
        Input grayscale image with values in [0, 1]
    se_size : int
        Size of the square structuring element (default: 5)
    clip_val : float
        Maximum gradient value before normalization (default: 1.0)
        Lower values increase sensitivity to faint edges.

    Returns
    -------
    np.ndarray
        Edge magnitude map with values in [0, 1]
    """
    # Create a square structuring element
    se = square(se_size)

    # Dilation: replace each pixel with the maximum in its neighborhood
    dilated = dilation(img, se)

    # Erosion: replace each pixel with the minimum in its neighborhood
    eroded = erosion(img, se)

    # The difference highlights edges (regions of rapid change)
    gradient = (dilated - eroded).astype(np.float32)

    # Normalize to [0, 1] using fixed scaling
    gradient = np.clip(gradient, 0.0, float(clip_val))
    gradient = gradient / float(clip_val)

    return gradient

## 4. The Sobel Operator (Reference)

The Sobel operator is a classical edge detector that estimates image gradients using
two 3×3 convolution kernels:

**Sobel-X** (detects vertical edges by measuring horizontal gradient):
```
[-1  0  +1]
[-2  0  +2]
[-1  0  +1]
```

**Sobel-Y** (detects horizontal edges by measuring vertical gradient):
```
[-1  -2  -1]
[ 0   0   0]
[+1  +2  +1]
```

The gradient magnitude is: $M = |G_x| + |G_y|$

We'll compare our learned filters to these classical kernels.

In [ ]:
# Define the standard Sobel kernels for reference
SOBEL_X = np.array([
    [-1, 0, +1],
    [-2, 0, +2],
    [-1, 0, +1]
], dtype=np.float32)

SOBEL_Y = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [+1, +2, +1]
], dtype=np.float32)


def sobel_magnitude(img: np.ndarray, clip_val: float = 1.0) -> np.ndarray:
    """
    Compute Sobel edge magnitude: |Gx| + |Gy| (L1 norm).

    Uses cross-correlation (same convention as PyTorch Conv2d).

    Parameters
    ----------
    img : np.ndarray
        Input grayscale image with values in [0, 1], shape (H, W)
    clip_val : float
        Maximum magnitude before normalization (default: 1.0)

    Returns
    -------
    np.ndarray
        Edge magnitude map with values in [0, 1], shape (H, W)
    """
    gx = ndcorrelate(img, SOBEL_X).astype(np.float32)
    gy = ndcorrelate(img, SOBEL_Y).astype(np.float32)
    mag = np.abs(gx) + np.abs(gy)
    mag = np.clip(mag, 0.0, float(clip_val))
    mag = mag / float(clip_val)
    return mag


def visualize_sobel_kernels():
    """Display the Sobel kernels with their values."""
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    fig.patch.set_facecolor(BG)

    for ax, kernel, name in [(axes[0], SOBEL_X, "Sobel-X\n(Vertical Edges)"),
                              (axes[1], SOBEL_Y, "Sobel-Y\n(Horizontal Edges)")]:
        vmax = np.abs(kernel).max()
        im = ax.imshow(kernel, cmap=MPL_EDGES_DIV, vmin=-vmax, vmax=vmax)

        # Add grid lines between pixels
        for i in range(4):
            ax.axhline(i - 0.5, color='black', linewidth=1)
            ax.axvline(i - 0.5, color='black', linewidth=1)

        # Annotate with actual values
        for i in range(3):
            for j in range(3):
                val = kernel[i, j]
                color = 'white' if abs(val) > 1 else 'black'
                ax.text(j, i, f'{val:+.0f}', ha='center', va='center',
                       fontsize=14, fontweight='bold', color=color)

        ax.set_title(name, fontsize=12, fontweight='bold', color='white')
        ax.set_xticks([])
        ax.set_yticks([])
        cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cb.ax.yaxis.set_tick_params(color='white')
        plt.setp(cb.ax.yaxis.get_ticklabels(), color='white')

    plt.suptitle("Classical Sobel Edge Detection Kernels", fontsize=14,
                 fontweight='bold', color='white')
    plt.tight_layout()
    plt.show()

visualize_sobel_kernels()

## 5. Dataset: Random Patch Sampling

Our training data consists of random patches extracted from source images.
For each patch, we compute the Beucher gradient as the target edge map.

### Data Augmentation

To help the network generalize, we apply random augmentations:
- **Horizontal flip** (50% chance)
- **Vertical flip** (50% chance)
- **90° rotations** (50% chance)
- **Gaussian noise** (added to input only, not target)

The noise is added only to the input, not the target. This teaches the network
to produce clean edge maps even from noisy inputs.

In [ ]:
class RandomPatchDataset(Dataset):
    """
    Dataset that extracts random patches from source images.

    For each sample:
    1. Randomly select one of the source images
    2. Extract a random PATCH_SIZE × PATCH_SIZE region from both the image
       and its pre-computed edge target
    3. Apply random augmentations (flip, rotate) to both patches
    4. Optionally add noise to the input (but not the target)

    Edge targets (Beucher or Sobel magnitude) are pre-computed once during
    initialization, eliminating redundant computation during training batches.

    This approach gives us effectively unlimited training data from a small
    set of source images.
    """

    def __init__(self, data_dir, patch_size=256, length=1000, augment=True):
        """
        Parameters
        ----------
        data_dir : Path or str
            Directory containing source images (jpg, jpeg, or png)
        patch_size : int
            Size of patches to extract
        length : int
            Number of samples per epoch (virtual dataset size)
        augment : bool
            Whether to apply random augmentations
        """
        self.data_dir = Path(data_dir)
        self.patch_size = patch_size
        self.length = length
        self.augment = augment

        # Find all image files
        self.image_paths = []
        for ext in ["*.jpg", "*.jpeg", "*.png"]:
            self.image_paths.extend(list(self.data_dir.glob(ext)))
        self.image_paths = sorted(self.image_paths)

        print(f"Found {len(self.image_paths)} source images in {data_dir}")
        if len(self.image_paths) == 0:
            raise ValueError(f"No images found in {data_dir}")

        # Pre-load all images into memory as float32 [0, 1]
        # and pre-compute edge magnitude targets
        self.images = []
        self.gradients = []
        target_label = TARGET_MODE
        for p in tqdm(self.image_paths, desc=f"Loading images and computing {TARGET_MODE} targets"):
            img = Image.open(p).convert("L")  # Convert to grayscale
            arr = np.asarray(img).astype(np.float32) / 255.0
            self.images.append(arr)
            # Pre-compute edge target for this image
            if TARGET_MODE == "beucher":
                grad = beucher_gradient(arr, se_size=SE_SIZE, clip_val=BEUCHER_CLIP)
            elif TARGET_MODE == "sobel":
                grad = sobel_magnitude(arr, clip_val=BEUCHER_CLIP)
            else:
                raise ValueError(f"Unknown TARGET_MODE: {TARGET_MODE}")
            self.gradients.append(grad)

        print(f"Pre-computed {target_label} targets for {len(self.images)} images")

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        # Randomly select a source image
        img_idx = np.random.randint(0, len(self.images))
        image = self.images[img_idx]
        gradient = self.gradients[img_idx]
        h, w = image.shape

        # If image is too small, resize it (and its gradient)
        if h < self.patch_size or w < self.patch_size:
            scale = max(self.patch_size / h, self.patch_size / w) * 1.1
            new_h, new_w = int(h * scale), int(w * scale)
            image = resize(image, (new_h, new_w), anti_aliasing=True).astype(np.float32)
            gradient = resize(gradient, (new_h, new_w), anti_aliasing=True).astype(np.float32)
            h, w = image.shape

        # Extract a random patch from both image and pre-computed gradient
        top = np.random.randint(0, h - self.patch_size + 1)
        left = np.random.randint(0, w - self.patch_size + 1)
        patch = image[top:top + self.patch_size, left:left + self.patch_size].copy()
        target = gradient[top:top + self.patch_size, left:left + self.patch_size].copy()

        # Apply augmentations to both patch and target
        if self.augment:
            # Random horizontal flip
            if np.random.random() > 0.5:
                patch = np.fliplr(patch).copy()
                target = np.fliplr(target).copy()
            # Random vertical flip
            if np.random.random() > 0.5:
                patch = np.flipud(patch).copy()
                target = np.flipud(target).copy()
            # Random 90° rotation
            if np.random.random() > 0.5:
                k = np.random.choice([1, 2, 3])  # 90°, 180°, or 270°
                patch = np.rot90(patch, k).copy()
                target = np.rot90(target, k).copy()

        # Add noise to INPUT only (not target)
        # This teaches the network to denoise while detecting edges
        input_patch = patch.copy()
        if ADD_INPUT_NOISE and NOISE_STD > 0:
            noise = np.random.normal(0, NOISE_STD, input_patch.shape).astype(np.float32)
            input_patch = np.clip(input_patch + noise, 0, 1)

        # Convert to PyTorch tensors with channel dimension
        x = torch.from_numpy(input_patch).unsqueeze(0)  # (1, H, W)
        y = torch.from_numpy(target).unsqueeze(0)        # (1, H, W)

        return x, y

## 6. Visualize Dataset Samples

Let's see what our training data looks like: input patches and their corresponding
Beucher gradient targets.

In [ ]:
def visualize_dataset_samples(data_dir=DATA_DIR, num_samples=4):
    """Display sample input-output pairs from the dataset."""
    ds = RandomPatchDataset(data_dir, patch_size=PATCH_SIZE, length=num_samples, augment=True)

    input_title = 'Input (with noise)' if ADD_INPUT_NOISE else 'Input'
    target_title = f'Target: {TARGET_MODE.title()} magnitude'

    for i in range(num_samples):
        x, y = ds[i]
        fig = make_subplots(rows=1, cols=2,
                            subplot_titles=[input_title, target_title],
                            horizontal_spacing=0.05)
        fig.add_trace(go.Heatmap(z=x.numpy()[0], colorscale='gray',
                                  zmin=0, zmax=1, showscale=False), row=1, col=1)
        fig.add_trace(go.Heatmap(z=y.numpy()[0], colorscale=CMAP_EDGES_SEQ,
                                  zmin=0, zmax=1), row=1, col=2)
        _square_sync(fig, 2)
        fig.update_layout(**DARK_LAYOUT, height=450, width=1000,
                          title_text=f'Training Sample {i+1}')
        fig.show()

visualize_dataset_samples(DATA_DIR, num_samples=4)

## 7. The Model: A Minimal Edge-Detecting CNN

Our network is intentionally minimal to make the learned filters directly interpretable.

### Architecture

```
Input (1×H×W)
     ↓
Conv2d(1→2, 3×3, padding=1)    ← Two learnable 3×3 filters (no bias)
     ↓
gx, gy (2×H×W)                 ← Signed filter responses
     ↓
|gx| + |gy|                    ← Gradient magnitude (L1 norm)
     ↓
clamp(mag / MAG_SCALE, 0, 1)   ← Scale to [0, 1]
     ↓
Output (1×H×W)
```

### Why This Architecture?

1. **Two filters** — We expect to learn two gradient directions (like Sobel-X and Sobel-Y)
2. **3×3 kernels** — Same size as Sobel, captures local gradients
3. **No activation function** — Raw filter outputs can be positive or negative (signed gradients)
4. **No bias** — Flat (constant) regions produce exactly zero response
5. **L1 magnitude** — Combines both directions: $|g_x| + |g_y|$
6. **Only 18 parameters** — 2 filters × 9 weights, easy to visualize and interpret

### What We Expect to Learn

If our network learns correctly, the two 3×3 filters should resemble gradient operators
that respond to edges in two different (ideally orthogonal) directions. They might look
like Sobel-X and Sobel-Y, or rotated versions (like 45° diagonals).

In [ ]:
class SimpleEdgeCNN(nn.Module):
    """
    A minimal CNN for learning edge detection.

    This network learns two 3×3 convolution filters that, when combined via
    L1 magnitude, predict edge strength. The filters are directly interpretable
    as gradient operators.

    The architecture is:
        Input → Conv2d(1→2, 3×3) → gx, gy → |gx| + |gy| → scale → Output

    With only 18 parameters (2 filters × 9 weights, no bias), this is one of the
    simplest possible learned edge detectors.
    """

    def __init__(self, mag_mode="l1", bias=False):
        """
        Parameters
        ----------
        mag_mode : str
            How to combine gx and gy:
            - "l1": |gx| + |gy| — Manhattan magnitude (default)
            - "l2": sqrt(gx² + gy²) — Euclidean magnitude
        bias : bool
            Whether to include a bias term. Disabled by default so that
            flat (constant) image regions produce exactly zero output.
        """
        super().__init__()
        self.mag_mode = mag_mode

        # Single convolutional layer: 1 input channel → 2 output channels
        # Each output channel is the result of convolving the input with a learned 3×3 filter
        # We use padding=1 to keep the output the same size as the input
        # Bias is disabled so flat regions map to exactly zero
        self.conv = nn.Conv2d(
            in_channels=1,
            out_channels=2,
            kernel_size=3,
            padding=1,
            bias=bias 
        )

        # Use default Kaiming initialization (std ≈ 0.47 for 3×3, 1-channel input).
        # This gives filter outputs large enough for the L1 magnitude gradient
        # (sign function) to be stable from the start.

    def forward(self, x):
        """
        Forward pass: compute edge magnitude from input image.

        Parameters
        ----------
        x : torch.Tensor
            Input images, shape (batch, 1, height, width), values in [0, 1]

        Returns
        -------
        edge : torch.Tensor
            Edge magnitude maps, shape (batch, 1, height, width), values in [0, 1]
        g : torch.Tensor
            Raw filter responses, shape (batch, 2, height, width)
            g[:, 0] = gx (first filter), g[:, 1] = gy (second filter)
        """
        # Apply the two learned 3×3 filters
        # Output shape: (batch, 2, height, width)
        g = self.conv(x)

        # Split into the two gradient components
        gx = g[:, 0:1, :, :]  # First filter response (like Sobel-X)
        gy = g[:, 1:2, :, :]  # Second filter response (like Sobel-Y)

        # Compute gradient magnitude
        if self.mag_mode == "l1":
            # Manhattan magnitude: |gx| + |gy|
            mag = torch.abs(gx) + torch.abs(gy)
        elif self.mag_mode == "l2":
            # Euclidean magnitude: sqrt(gx² + gy²)
            # The small epsilon (1e-8) prevents NaN gradients when gx=gy=0
            mag = torch.sqrt(gx * gx + gy * gy + 1e-8)
        else:
            raise ValueError(f"Unknown mag_mode: {self.mag_mode}. Use 'l1' or 'l2'.")

        # Scale to [0, 1] range
        # MAG_SCALE controls the expected magnitude range
        edge = torch.clamp(mag / MAG_SCALE, 0, 1)

        return edge, g

    def get_filters(self):
        """
        Extract the learned 3×3 filters for visualization.

        Returns
        -------
        np.ndarray
            Array of shape (2, 3, 3) containing the two learned filters
        """
        # conv.weight has shape (out_channels, in_channels, H, W) = (2, 1, 3, 3)
        w = self.conv.weight.detach().cpu().numpy()
        return w[:, 0, :, :].copy()  # Remove the input channel dimension → (2, 3, 3)

In [ ]:
# Set random seed for reproducible weight initialization
torch.manual_seed(42)

# Create the model
model = SimpleEdgeCNN(mag_mode=MAG_MODE).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print("Model Architecture:")
print("=" * 50)
print(model)
print("=" * 50)
print(f"Total parameters: {total_params}")
print(f"  - conv.weight: 2 filters × 1 channel × 3 × 3 = 18 parameters")
print(f"  - conv.bias: disabled (flat regions → zero response)")
print("=" * 50)

## 8. Create Data Loaders

In [ ]:
# Training dataset: random patches with augmentation
train_dataset = RandomPatchDataset(
    DATA_DIR,
    patch_size=PATCH_SIZE,
    length=PATCHES_PER_EPOCH,
    augment=True
)

# Validation dataset: random patches without augmentation
# Smaller than training set (20%)
val_dataset = RandomPatchDataset(
    DATA_DIR,
    patch_size=PATCH_SIZE,
    length=PATCHES_PER_EPOCH // 5,
    augment=False
)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Training batches per epoch: {len(train_loader)}")
print(f"Validation batches per epoch: {len(val_loader)}")

## 9. Loss Function and Optimizer

### Loss Function

We use **Soft Dice Loss**, an overlap-based metric:

$\displaystyle \mathcal{L} = 1 - \frac{2 \sum_i \hat{y}_i \, y_i}{\sum_i \hat{y}_i^2 + \sum_i y_i^2}$

Unlike per-pixel losses (L1, MSE), Dice measures the overlap between predicted and target
edge maps. This is important because edge pixels are rare compared to background — a naive
per-pixel loss could achieve low error by simply predicting "no edge" everywhere. Dice
naturally handles this class imbalance.

### Optimizer

We use **Adam**, which adapts the learning rate per-parameter. This is particularly
helpful here because the gradient of $|g_x| + |g_y|$ involves the sign function,
which has zero gradient when the filter response is exactly zero. Adam's momentum
helps push through these flat regions early in training.

### Learning Rate Scheduler

We use **Cosine Annealing**: the learning rate smoothly decays from its initial value
down to a minimum (`eta_min=1e-6`) following a cosine curve over the full training run.
This avoids the staircase drops of plateau-based schedulers and provides predictable,
smooth decay — well suited to a fixed epoch budget.

In [ ]:
class SoftDiceLoss(nn.Module):
    """Soft Dice loss for continuous predictions and targets in [0, 1].

    Dice = 2 * sum(pred * target) / (sum(pred²) + sum(target²))

    Unlike L1/MSE, this measures overlap rather than per-pixel error,
    so it isn't dominated by the large background (non-edge) region.
    """
    def __init__(self, eps=1e-7):
        super().__init__()
        self.eps = eps

    def forward(self, pred, target):
        intersection = (pred * target).sum()
        denominator = (pred * pred).sum() + (target * target).sum()
        dice = (2.0 * intersection + self.eps) / (denominator + self.eps)
        return 1.0 - dice


def make_criterion():
    """Create the loss function based on LOSS_MODE setting."""
    if LOSS_MODE == "l1":
        return nn.L1Loss()
    elif LOSS_MODE == "mse":
        return nn.MSELoss()
    elif LOSS_MODE == "dice":
        return SoftDiceLoss()
    else:
        raise ValueError(f"Unknown LOSS_MODE: {LOSS_MODE}. Use 'l1', 'mse', or 'dice'.")

criterion = make_criterion()
print(f"Loss function: {criterion}")

# Adam optimizer — adapts learning rate per-parameter
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
print(f"Optimizer: Adam(lr={LEARNING_RATE})")

# Cosine annealing: smooth LR decay over the full training run
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)
print(f"Scheduler: CosineAnnealingLR(T_max={NUM_EPOCHS}, eta_min=1e-6)")

## 10. Training Utilities

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    """
    Train the model for one epoch.

    Returns
    -------
    float
        Average training loss for this epoch
    """
    model.train()
    running_loss = 0.0
    num_samples = 0

    pbar = tqdm(loader, leave=False, desc="Training")
    for x, y in pbar:
        x = x.to(device)
        y = y.to(device)

        # Forward pass
        optimizer.zero_grad(set_to_none=True)
        y_hat, _ = model(x)

        # Compute loss
        loss = criterion(y_hat, y)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Accumulate statistics
        running_loss += loss.item() * x.size(0)
        num_samples += x.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / max(num_samples, 1)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """
    Evaluate the model on a dataset.

    Returns
    -------
    float
        Average loss on the dataset
    """
    model.eval()
    running_loss = 0.0
    num_samples = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        y_hat, _ = model(x)
        loss = criterion(y_hat, y)

        running_loss += loss.item() * x.size(0)
        num_samples += x.size(0)

    return running_loss / max(num_samples, 1)

## 11. Visualization Functions

In [ ]:
@torch.no_grad()
def visualize_predictions(model, dataloader, device, num_samples=3):
    """
    Visualize model predictions alongside ground truth.

    Shows for each sample:
    - Input image (grayscale)
    - Target edge magnitude (sequential edge cmap)
    - Predicted edge map (sequential edge cmap)
    - Internal gx response (diverging edge cmap)
    - Internal gy response (diverging edge cmap)
    """
    model.eval()
    x, y = next(iter(dataloader))
    x = x.to(device)

    y_hat, g = model(x)

    # Move to CPU for plotting
    x_np = x.cpu().numpy()
    y_np = y.numpy()
    y_hat_np = y_hat.cpu().numpy()
    g_np = g.cpu().numpy()

    num_samples = min(num_samples, len(x_np))
    target_label = f'Target ({TARGET_MODE.title()})'

    for i in range(num_samples):
        img = x_np[i, 0]
        tgt = y_np[i, 0]
        pred = y_hat_np[i, 0]
        gx = g_np[i, 0]
        gy = g_np[i, 1]

        vmax_gx = max(abs(float(gx.min())), abs(float(gx.max())), 1e-8)
        vmax_gy = max(abs(float(gy.min())), abs(float(gy.max())), 1e-8)

        fig = make_subplots(rows=1, cols=5,
                            subplot_titles=['Input', target_label,
                                            'Prediction', 'Filter 1 (gx)',
                                            'Filter 2 (gy)'],
                            horizontal_spacing=0.03)
        fig.add_trace(go.Heatmap(z=img, colorscale='gray',
                                  zmin=0, zmax=1, showscale=False), row=1, col=1)
        fig.add_trace(go.Heatmap(z=tgt, colorscale=CMAP_EDGES_SEQ,
                                  zmin=0, zmax=1, showscale=False), row=1, col=2)
        fig.add_trace(go.Heatmap(z=pred, colorscale=CMAP_EDGES_SEQ,
                                  zmin=0, zmax=1, showscale=False), row=1, col=3)
        fig.add_trace(go.Heatmap(z=gx, colorscale=CMAP_EDGES_DIV,
                                  zmin=-vmax_gx, zmax=vmax_gx,
                                  showscale=False), row=1, col=4)
        fig.add_trace(go.Heatmap(z=gy, colorscale=CMAP_EDGES_DIV,
                                  zmin=-vmax_gy, zmax=vmax_gy), row=1, col=5)
        _square_sync(fig, 5)
        fig.update_layout(**DARK_LAYOUT, height=400, width=1600,
                          title_text=f'Sample {i+1}')
        fig.show()


def visualize_filters(model):
    """
    Visualize the learned 3×3 filters compared to Sobel kernels.
    """
    filters = model.get_filters()  # (2, 3, 3)

    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    fig.patch.set_facecolor(BG)

    # Top row: learned filters
    for i in range(2):
        f = filters[i]
        vmax = np.abs(f).max() + 1e-8
        im = axes[0, i].imshow(f, cmap=MPL_EDGES_DIV, vmin=-vmax, vmax=vmax)
        axes[0, i].set_title(f"Learned Filter #{i+1}", fontsize=12,
                              fontweight="bold", color='white')

        # Add grid lines
        for r in range(4):
            axes[0, i].axhline(r - 0.5, color='black', linewidth=1)
            axes[0, i].axvline(r - 0.5, color='black', linewidth=1)

        # Annotate with values
        for r in range(3):
            for c in range(3):
                val = f[r, c]
                color = 'white' if abs(val) > vmax * 0.5 else 'black'
                axes[0, i].text(c, r, f'{val:.2f}', ha='center', va='center',
                               fontsize=10, color=color)

        axes[0, i].set_xticks([])
        axes[0, i].set_yticks([])
        cb = fig.colorbar(im, ax=axes[0, i], fraction=0.046, pad=0.04)
        cb.ax.yaxis.set_tick_params(color='white')
        plt.setp(cb.ax.yaxis.get_ticklabels(), color='white')

    # Bottom row: Sobel reference
    for i, (sobel, name) in enumerate([(SOBEL_X, "Sobel-X (Reference)"),
                                        (SOBEL_Y, "Sobel-Y (Reference)")]):
        vmax = np.abs(sobel).max() + 1e-8
        im = axes[1, i].imshow(sobel, cmap=MPL_EDGES_DIV, vmin=-vmax, vmax=vmax)
        axes[1, i].set_title(name, fontsize=12, color='white')

        for r in range(4):
            axes[1, i].axhline(r - 0.5, color='black', linewidth=1)
            axes[1, i].axvline(r - 0.5, color='black', linewidth=1)

        for r in range(3):
            for c in range(3):
                val = sobel[r, c]
                color = 'white' if abs(val) > 1 else 'black'
                axes[1, i].text(c, r, f'{val:+.0f}', ha='center', va='center',
                               fontsize=12, fontweight='bold', color=color)

        axes[1, i].set_xticks([])
        axes[1, i].set_yticks([])
        cb = fig.colorbar(im, ax=axes[1, i], fraction=0.046, pad=0.04)
        cb.ax.yaxis.set_tick_params(color='white')
        plt.setp(cb.ax.yaxis.get_ticklabels(), color='white')

    plt.suptitle("Learned Filters vs Sobel Reference", fontsize=14,
                 fontweight="bold", color='white')
    plt.tight_layout()
    plt.show()

    # Print filter values
    print("\nLearned Filter Weights:")
    print("=" * 50)
    for i, f in enumerate(filters):
        print(f"Filter #{i+1}:")
        for row in f:
            print(f"  [{row[0]:+.3f}, {row[1]:+.3f}, {row[2]:+.3f}]")
        print()

## 12. Frequency Response Analysis

### What is a Frequency Response?

Every linear filter (like our 3×3 convolution kernels) can be characterized by its
**frequency response**: how it amplifies or attenuates different spatial frequencies.

- **Low frequencies** = smooth, slowly varying regions (center of the plot)
- **High frequencies** = rapid changes, fine details, edges (edges of the plot)

### Edge Detectors in Frequency Domain

Edge-detecting filters like Sobel are **high-pass filters**: they suppress low frequencies
(smooth regions) and amplify high frequencies (edges).

The frequency response is computed by:
1. Zero-padding the 3×3 kernel to a larger size (e.g., 64×64)
2. Computing the 2D FFT
3. Shifting so DC (zero frequency) is at the center
4. Taking the magnitude

### What to Look For

- **Sobel-X**: Strong response along the horizontal frequency axis (detects vertical edges)
- **Sobel-Y**: Strong response along the vertical frequency axis (detects horizontal edges)
- **Learned filters**: Should show similar high-pass characteristics, possibly rotated

In [ ]:
def compute_frequency_response(kernel, pad_size=64):
    """
    Compute the 2D frequency response (magnitude spectrum) of a filter kernel.

    Parameters
    ----------
    kernel : np.ndarray
        The filter kernel, shape (3, 3)
    pad_size : int
        Size to zero-pad to (controls resolution of frequency plot)

    Returns
    -------
    np.ndarray
        Magnitude spectrum, shape (pad_size, pad_size)
    """
    # Zero-pad the kernel to get better frequency resolution
    padded = np.zeros((pad_size, pad_size), dtype=np.float64)
    padded[:kernel.shape[0], :kernel.shape[1]] = kernel

    # Compute 2D FFT
    fft = np.fft.fft2(padded)

    # Shift so DC is at center
    fft_shifted = np.fft.fftshift(fft)

    # Return magnitude
    return np.abs(fft_shifted)


def visualize_frequency_responses(model, pad_size=64):
    """
    Compare frequency responses of learned filters vs Sobel kernels.
    """
    filters = model.get_filters()  # (2, 3, 3)

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.patch.set_facecolor(BG)

    all_filters = [
        (SOBEL_X, "Sobel-X"),
        (SOBEL_Y, "Sobel-Y"),
        (filters[0], "Learned Filter #1"),
        (filters[1], "Learned Filter #2"),
    ]

    positions = [(0, 0), (0, 1), (1, 0), (1, 1)]

    for (kernel, name), (row, col) in zip(all_filters, positions):
        freq_response = compute_frequency_response(kernel, pad_size)

        # Use log scale for better visualization
        freq_response_log = np.log1p(freq_response)

        im = axes[row, col].imshow(
            freq_response_log,
            cmap=MPL_EDGES_DIV,
            extent=[-0.5, 0.5, -0.5, 0.5]
        )
        axes[row, col].set_title(name, fontsize=12, fontweight="bold",
                                  color='white')
        axes[row, col].set_xlabel("Frequency X (cycles/pixel)", color='white')
        axes[row, col].set_ylabel("Frequency Y (cycles/pixel)", color='white')
        axes[row, col].tick_params(colors='white')
        cb = fig.colorbar(im, ax=axes[row, col], fraction=0.046, pad=0.04)
        cb.set_label("log(1 + |H(f)|)", color='white')
        cb.ax.yaxis.set_tick_params(color='white')
        plt.setp(cb.ax.yaxis.get_ticklabels(), color='white')

    plt.suptitle("Frequency Responses: Sobel vs Learned Filters", fontsize=14,
                 fontweight="bold", color='white')
    plt.tight_layout()
    plt.show()

    # Side-by-side comparison
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    fig.patch.set_facecolor(BG)

    for ax, (kernel, name) in zip(axes, all_filters):
        freq_response = compute_frequency_response(kernel, pad_size)
        freq_response_log = np.log1p(freq_response)

        ax.imshow(freq_response_log, cmap=MPL_EDGES_DIV,
                  extent=[-0.5, 0.5, -0.5, 0.5])
        ax.set_title(name, fontsize=11, fontweight="bold", color='white')
        ax.set_xlabel("Freq X", color='white')
        ax.set_ylabel("Freq Y", color='white')
        ax.tick_params(colors='white')

    plt.suptitle("Frequency Response Comparison (All Filters)", fontsize=14,
                 fontweight="bold", color='white')
    plt.tight_layout()
    plt.show()

## 13. Visualize Initial State (Before Training)

Let's see what the randomly initialized filters look like before any training.

In [ ]:
print("=" * 60)
print("BEFORE TRAINING")
print("=" * 60)
print("\nRandomly initialized filters:")
visualize_filters(model)

print("\nPredictions with random filters:")
visualize_predictions(model, val_loader, DEVICE, num_samples=2)

print("\nFrequency responses of random filters:")
visualize_frequency_responses(model)

## 14. Training Loop

Now we train the network to minimize the difference between its predictions
and the Beucher gradient targets.

In [ ]:
import copy

history = {"train": [], "val": []}
state_history = []  # model state_dict after each epoch
best_val_loss = float("inf")

print("=" * 60)
print("TRAINING")
print("=" * 60)
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Loss function: {LOSS_MODE}")
print("=" * 60)

for epoch in range(NUM_EPOCHS):
    # Train for one epoch
    train_loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)

    # Evaluate on validation set
    val_loss = evaluate(model, val_loader, criterion, DEVICE)

    # Step the cosine annealing scheduler
    scheduler.step()

    # Record history
    history["train"].append(train_loss)
    history["val"].append(val_loss)

    # Store a snapshot of the model weights
    state_history.append(copy.deepcopy(model.state_dict()))

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_edge_detector_beucher.pth")

    # Print progress
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS}  "
          f"Train: {train_loss:.6f}  Val: {val_loss:.6f}  LR: {current_lr:.6f}")

    # Visualize progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        visualize_predictions(model, val_loader, DEVICE, num_samples=2)
        visualize_filters(model)

print("\n" + "=" * 60)
print(f"Training complete! Best validation loss: {best_val_loss:.6f}")
print(f"Saved {len(state_history)} filter snapshots for exploration.")
print("=" * 60)

## 15. Training History

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=history["train"], mode='lines', name='Train Loss',
                          line=dict(width=2, color='#2ec4b6')))
fig.add_trace(go.Scatter(y=history["val"], mode='lines', name='Validation Loss',
                          line=dict(width=2, color='#d4702a')))
fig.update_layout(**DARK_LAYOUT, width=800, height=400,
                  title='Training History',
                  xaxis_title='Epoch',
                  yaxis_title=f'Loss ({LOSS_MODE.upper()})',
                  xaxis=dict(gridcolor='rgba(255,255,255,0.1)'),
                  yaxis=dict(gridcolor='rgba(255,255,255,0.1)'))
fig.show()

print(f"Final train loss: {history['train'][-1]:.6f}")
print(f"Final val loss: {history['val'][-1]:.6f}")
print(f"Best val loss: {best_val_loss:.6f}")

## 16. Final Results

Now let's examine what the network learned after training.

In [ ]:
# Load best model
model.load_state_dict(torch.load("best_edge_detector_beucher.pth"))

print("=" * 60)
print("AFTER TRAINING: LEARNED FILTERS")
print("=" * 60)

# Visualize learned filters
visualize_filters(model)

## 17. Frequency Response Analysis (After Training)

Let's compare the frequency responses of our learned filters to the Sobel kernels.

**Key questions:**
1. Do the learned filters act as high-pass filters (like edge detectors should)?
2. Are they sensitive to similar frequency bands as Sobel?
3. Do they show orthogonal directional preferences?

In [ ]:
print("=" * 60)
print("FREQUENCY RESPONSE ANALYSIS")
print("=" * 60)

visualize_frequency_responses(model)

## 18. Final Predictions

Let's see the final model predictions on some test samples.

In [ ]:
print("=" * 60)
print("FINAL PREDICTIONS")
print("=" * 60)

visualize_predictions(model, val_loader, DEVICE, num_samples=4)

## 19. Summary and Key Takeaways

### What We Built

A minimal CNN with only **18 parameters** (2 filters × 9 weights, no bias) that learns
to detect edges in images.

### What We Learned

1. **Convolution as pattern matching**: Each 3×3 filter responds to a specific local pattern.
   Gradient operators respond when there's a change in intensity across the kernel.

2. **Classical vs learned filters**: The Sobel operator is hand-designed based on calculus
   (discrete derivatives). Our network discovers similar filters automatically through
   gradient descent.

3. **Magnitude from components**: By combining two directional filters via L1 norm
   ($|g_x| + |g_y|$), we get an edge strength measure that responds to edges in any direction.

4. **Frequency domain interpretation**: Edge detectors are high-pass filters that
   suppress smooth regions and enhance rapid changes.

### Comparison: Learned vs Sobel

| Aspect | Sobel | Learned |
|--------|-------|---------|
| Design | Hand-crafted | Learned from data |
| Parameters | Fixed | Optimized for the task |
| Directions | Axis-aligned (X, Y) | May be rotated |
| Flexibility | None | Adapts to target |

### Why This Matters

This simple example illustrates the core idea behind modern computer vision:

> Instead of hand-designing feature detectors, let neural networks learn them from data.

In larger CNNs (VGG, ResNet, etc.), the first layers typically learn edge and texture
detectors very similar to what we found here. Deeper layers then combine these simple
features into increasingly complex patterns.

## 20. Explore Filter Evolution

Use the slider below to step through epochs and see how the learned filters
and predictions evolve during training.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Grab a fixed batch for consistent comparison across epochs
_explore_x, _explore_y = next(iter(val_loader))
_explore_x = _explore_x.to(DEVICE)

def show_epoch(epoch_num):
    """Load model state from a given epoch and display filters + predictions."""
    idx = epoch_num - 1
    model.load_state_dict(state_history[idx])
    model.eval()

    filters = model.get_filters()  # (2, 3, 3)

    with torch.no_grad():
        y_hat, g = model(_explore_x)

    x_np = _explore_x.cpu().numpy()
    y_np = _explore_y.numpy()
    y_hat_np = y_hat.cpu().numpy()
    g_np = g.cpu().numpy()

    train_loss = history["train"][idx]
    val_loss = history["val"][idx]

    # --- Row 1: Learned filters + Sobel reference ---
    fig_filters, axes = plt.subplots(1, 4, figsize=(16, 3.5))
    fig_filters.patch.set_facecolor(BG)

    all_k = [
        (filters[0], "Learned #1"),
        (filters[1], "Learned #2"),
        (SOBEL_X, "Sobel-X (ref)"),
        (SOBEL_Y, "Sobel-Y (ref)"),
    ]
    for ax, (kernel, name) in zip(axes, all_k):
        vmax = max(np.abs(kernel).max(), 1e-8)
        im = ax.imshow(kernel, cmap=MPL_EDGES_DIV, vmin=-vmax, vmax=vmax)
        for r in range(4):
            ax.axhline(r - 0.5, color='black', linewidth=1)
            ax.axvline(r - 0.5, color='black', linewidth=1)
        for r in range(3):
            for c in range(3):
                val = kernel[r, c]
                color = 'white' if abs(val) > vmax * 0.5 else 'black'
                ax.text(c, r, f'{val:.2f}', ha='center', va='center',
                        fontsize=10, color=color)
        ax.set_title(name, fontsize=11, fontweight='bold', color='white')
        ax.set_xticks([])
        ax.set_yticks([])
        cb = fig_filters.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cb.ax.yaxis.set_tick_params(color='white')
        plt.setp(cb.ax.yaxis.get_ticklabels(), color='white')

    fig_filters.suptitle(
        f"Epoch {epoch_num}/{len(state_history)}  —  "
        f"Train loss: {train_loss:.5f}  Val loss: {val_loss:.5f}",
        fontsize=13, fontweight='bold', color='white')
    plt.tight_layout()
    plt.show()

    # --- Row 2: Predictions on two samples ---
    for i in range(min(2, len(x_np))):
        img = x_np[i, 0]
        tgt = y_np[i, 0]
        pred = y_hat_np[i, 0]
        gx = g_np[i, 0]
        gy = g_np[i, 1]
        vmax_gx = max(abs(float(gx.min())), abs(float(gx.max())), 1e-8)
        vmax_gy = max(abs(float(gy.min())), abs(float(gy.max())), 1e-8)

        fig = make_subplots(rows=1, cols=5,
                            subplot_titles=['Input', f'Target ({TARGET_MODE.title()})',
                                            'Prediction', 'Filter 1 (gx)',
                                            'Filter 2 (gy)'],
                            horizontal_spacing=0.03)
        fig.add_trace(go.Heatmap(z=img, colorscale='gray',
                                  zmin=0, zmax=1, showscale=False), row=1, col=1)
        fig.add_trace(go.Heatmap(z=tgt, colorscale=CMAP_EDGES_SEQ,
                                  zmin=0, zmax=1, showscale=False), row=1, col=2)
        fig.add_trace(go.Heatmap(z=pred, colorscale=CMAP_EDGES_SEQ,
                                  zmin=0, zmax=1, showscale=False), row=1, col=3)
        fig.add_trace(go.Heatmap(z=gx, colorscale=CMAP_EDGES_DIV,
                                  zmin=-vmax_gx, zmax=vmax_gx,
                                  showscale=False), row=1, col=4)
        fig.add_trace(go.Heatmap(z=gy, colorscale=CMAP_EDGES_DIV,
                                  zmin=-vmax_gy, zmax=vmax_gy), row=1, col=5)
        _square_sync(fig, 5)
        fig.update_layout(**DARK_LAYOUT, height=350, width=1600,
                          title_text=f'Sample {i+1}')
        fig.show()

    # Restore best model when done exploring
    # (uncomment the next line if you want auto-restore)
    # model.load_state_dict(torch.load("best_edge_detector_beucher.pth"))


epoch_slider = widgets.IntSlider(
    value=len(state_history),
    min=1,
    max=len(state_history),
    step=1,
    description='Epoch:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px'),
)

out = widgets.interactive_output(show_epoch, {'epoch_num': epoch_slider})
display(epoch_slider, out)